In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["numpy", "Pillow"]
ASSETS = [{"name": "img_test.png", "url": "/labs/3.2.2/img_test.png", "size": 16753, "sha256": "85dbd75a5b0c81491ddeae785cd208f4084b608bcd8b27deca87e3e6be473606"}]

if PACKAGES:
    await piplite.install(PACKAGES)

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    response = await pyfetch(asset["url"])
    payload = await response.bytes()
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 3.2.2 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
# JupyterLite 浏览器 ONNX Runtime：从现有分片清单加载、校验、缓存并执行模型
from pyodide.code import run_js

_browser_ort_run = run_js(r"""
async (taskCode) => {
  const origin = self.location.origin;
  const digest = async buffer => {
    const hash = await crypto.subtle.digest("SHA-256", buffer);
    return [...new Uint8Array(hash)].map(value => value.toString(16).padStart(2, "0")).join("");
  };
  const response = await fetch(`${origin}/labs/manifest.json`);
  if (!response.ok) throw new Error("模型清单加载失败");
  const manifest = await response.json();
  const model = manifest.tasks?.[taskCode]?.model;
  if (!model) throw new Error(`任务 ${taskCode} 没有浏览器模型`);

  const cache = await caches.open("ai-trainer-onnx-v2");
  const cacheKey = new Request(`${origin}/labs/model-cache/${model.modelId}.onnx`);
  let modelBytes;
  const cached = await cache.match(cacheKey);
  if (cached) {
    const cachedBuffer = await cached.arrayBuffer();
    if (await digest(cachedBuffer) === model.sha256) modelBytes = new Uint8Array(cachedBuffer);
  }
  if (!modelBytes) {
    const chunks = [];
    let total = 0;
    for (const part of model.chunks) {
      const partResponse = await fetch(`${origin}${part.url}`);
      if (!partResponse.ok) throw new Error(`模型分片加载失败：${part.url}`);
      const partBuffer = await partResponse.arrayBuffer();
      if (await digest(partBuffer) !== part.sha256) throw new Error(`模型分片校验失败：${part.url}`);
      const bytes = new Uint8Array(partBuffer);
      chunks.push(bytes);
      total += bytes.byteLength;
    }
    modelBytes = new Uint8Array(total);
    let offset = 0;
    for (const chunk of chunks) { modelBytes.set(chunk, offset); offset += chunk.byteLength; }
    if (await digest(modelBytes.buffer) !== model.sha256) throw new Error("模型完整性校验失败");
    await cache.put(cacheKey, new Response(modelBytes, { headers: { "content-type": "application/octet-stream" } }));
  }

  const runtimeBase = `${origin}/labs/runtime/onnxruntime-web/1.27.0/`;
  const ort = await import(`${runtimeBase}ort.wasm.bundle.min.js`);
  ort.env.wasm.numThreads = 1;
  ort.env.wasm.wasmPaths = runtimeBase;
  const session = await ort.InferenceSession.create(modelBytes, { executionProviders: ["wasm"] });
  const feeds = {};
  const inputs = [];
  for (const [index, name] of session.inputNames.entries()) {
    const meta = session.inputMetadata[index];
    if (!meta?.isTensor) continue;
    const shape = (meta.shape?.length ? meta.shape : [1]).map(value => Number.isInteger(value) && value > 0 ? value : 1);
    const size = shape.reduce((left, right) => left * right, 1);
    const sourceType = String(meta.type ?? "float32");
    const type = sourceType.includes("uint8") ? "uint8" : sourceType.includes("int32") ? "int32" : "float32";
    const data = type === "uint8" ? new Uint8Array(size) : type === "int32" ? new Int32Array(size) : new Float32Array(size);
    feeds[name] = new ort.Tensor(type, data, shape);
    inputs.push({ name, type, shape });
  }
  const outputMap = await session.run(feeds);
  const outputs = Object.entries(outputMap).map(([name, tensor]) => ({ name, type: tensor.type, shape: [...tensor.dims] }));
  return { model: model.name, runtime: model.browserRuntime, variant: model.sourceVariant, inputs, outputs };
}
""")

async def browser_onnx_check(task_code="3.2.2"):
    """执行一次真实的浏览器 WASM 推理，并返回张量结构。"""
    result = await _browser_ort_run(task_code)
    return result.to_py() if hasattr(result, "to_py") else result

onnx_result = await browser_onnx_check()
onnx_result


In [ ]:
import onnxruntime
import numpy as np
from PIL import Image


# 加载ONNX模型  2分
ort_session = __________________


# 加载图像 2分
image = __________________('L')  # 转为灰度图


#图像预处理 
image = __________________((28, 28))  # 调整大小为MNIST模型的输入尺寸2分
image_array = __________________(__________________, dtype=np.float32)  # 转为numpy数组2分
image_array = __________________(__________________, axis=0)  # 添加batch维度2分
image_array = __________________(__________________, axis=0)  # 添加通道维度2分


#返回模型输入列表 2分
ort_inputs = {__________________()[0].name: image_array}
# 执行预测 2分
ort_outs = __________________(None, ort_inputs)


# 获取预测结果 2分
predicted_class = __________________


# 输出预测结果
print(f"Predicted class: {predicted_class}")
